# 📊 Consumer Behavior: Variance & Standard Deviation Analysis
### Measuring Income Dispersion Across 11,789 Consumers

**Author:** Akmal Raxmatov  
**Dataset:** `data.csv` — 11,789 records, 25 features  
**Objective:** Understand the *spread* of monthly income using Variance, Standard Deviation, IQR, and the Empirical Rule.

> *"The mean tells you where the center is. The standard deviation tells you how much the data disagrees with that center."*

## 1. Theoretical Foundation

While central tendency (mean, median, mode) tells us *where* data clusters, **dispersion** tells us *how spread out* it is — critical for economic modeling, risk analysis, and policy design.

| Measure | Formula | What It Tells You |
|:---|:---:|:---|
| **Variance** | $s^2 = \\frac{\\sum(x_i - \\bar{x})^2}{n-1}$ | Average squared distance from the mean |
| **Std Deviation** | $s = \\sqrt{s^2}$ | Spread in the original unit (USD) |
| **IQR** | $Q_3 - Q_1$ | Spread of the middle 50% (outlier-resistant) |
| **CV** | $\\frac{s}{\\bar{x}} \\times 100\\%$ | Relative spread — comparable across datasets |

### The Empirical Rule (Normal Distribution)
If data is normally distributed:
- **±1σ** covers ~68.27% of all values
- **±2σ** covers ~95.45% of all values
- **±3σ** covers ~99.73% of all values

Comparing our dataset against these benchmarks reveals the true shape of income distribution.

## 2. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Load dataset
data = pd.read_csv('data.csv')
inc  = data['monthly_income'].dropna()

print(f'Total records:        {len(data):,}')
print(f'Income records (clean): {len(inc):,}')
print(f'Missing values:         {data["monthly_income"].isnull().sum()}')
data.head()

## 3. Computing Dispersion Measures

In [ ]:
# === Core dispersion metrics ===
mean_val = inc.mean()
std_val  = inc.std()          # Pandas uses ddof=1 (sample std) by default
var_val  = inc.var()          # Sample variance
cv       = (std_val / mean_val) * 100

# === Percentile-based measures ===
q1  = inc.quantile(0.25)
q3  = inc.quantile(0.75)
iqr = q3 - q1

# === Shape metrics ===
skewness = inc.skew()
kurtosis = inc.kurt()

print('=' * 45)
print('         DISPERSION ANALYSIS RESULTS')
print('=' * 45)
print(f'  Mean Income      : ${mean_val:>12,.2f}')
print(f'  Std Deviation    : ${std_val:>12,.2f}')
print(f'  Variance         :  {var_val:>12,.2f}')
print(f'  CV               :  {cv:>11.1f}%')
print(f'  IQR              : ${iqr:>12,.2f}')
print(f'  Q1               : ${q1:>12,.2f}')
print(f'  Q3               : ${q3:>12,.2f}')
print(f'  Skewness         :  {skewness:>12.4f}')
print(f'  Kurtosis         :  {kurtosis:>12.4f}')
print('=' * 45)

## 4. Empirical Rule Verification

In [ ]:
# Check what % of data falls within 1, 2, 3 standard deviations
within_1sigma = ((inc >= mean_val - std_val)  & (inc <= mean_val + std_val)).mean()  * 100
within_2sigma = ((inc >= mean_val - 2*std_val) & (inc <= mean_val + 2*std_val)).mean() * 100
within_3sigma = ((inc >= mean_val - 3*std_val) & (inc <= mean_val + 3*std_val)).mean() * 100

comparison = pd.DataFrame({
    'Range': ['Mean ± 1σ', 'Mean ± 2σ', 'Mean ± 3σ'],
    'Our Dataset (%)':     [f'{within_1sigma:.1f}%', f'{within_2sigma:.1f}%', f'{within_3sigma:.1f}%'],
    'Normal Dist. (%)':    ['68.27%', '95.45%', '99.73%'],
    'Lower Bound (USD)':   [f'${mean_val-std_val:,.0f}', f'${mean_val-2*std_val:,.0f}', f'${mean_val-3*std_val:,.0f}'],
    'Upper Bound (USD)':   [f'${mean_val+std_val:,.0f}', f'${mean_val+2*std_val:,.0f}', f'${mean_val+3*std_val:,.0f}'],
})
comparison

## 5. Visualization — 7-Panel Analysis

In [ ]:
import matplotlib.gridspec as gridspec

BG = '#0d1117'; PANEL = '#161b22'; BORDER = '#30363d'
TEXT = '#e6edf3'; MUTED = '#8b949e'
BLUE='#58a6ff'; ORANGE='#ffa657'; GREEN='#3fb950'
RED='#ff7b72'; PURPLE='#d2a8ff'; YELLOW='#e3b341'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': PANEL,
    'axes.edgecolor': BORDER, 'axes.labelcolor': MUTED,
    'xtick.color': MUTED, 'ytick.color': MUTED,
    'text.color': TEXT, 'grid.color': BORDER, 'grid.alpha': 0.5
})

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

fig.text(0.5, 0.97, 'Monthly Income — Variance & Standard Deviation Analysis',
         ha='center', fontsize=17, fontweight='bold', color=TEXT)
fig.text(0.5, 0.945, f'n = {len(inc):,}  ·  Author: Akmal Raxmatov',
         ha='center', fontsize=10, color=MUTED)

# Plot 1: Main histogram
ax1 = fig.add_subplot(gs[0, :])
n_counts, bins, _ = ax1.hist(inc, bins=40, color=BLUE, alpha=0.55, edgecolor=BG, linewidth=0.4)
ax1.axvspan(mean_val-std_val, mean_val+std_val, alpha=0.12, color=GREEN, label=f'±1σ ({within_1sigma:.1f}%)')
ax1.axvspan(mean_val-2*std_val, mean_val+2*std_val, alpha=0.07, color=YELLOW, label=f'±2σ ({within_2sigma:.1f}%)')
ax1.axvline(mean_val, color=RED,    lw=2,   linestyle='--', label=f'Mean  ${mean_val:,.0f}')
ax1.axvline(inc.median(), color=GREEN, lw=2, linestyle='-', label=f'Median ${inc.median():,.0f}')
ax1.axvline(mean_val+std_val, color=ORANGE, lw=1.5, linestyle=':', label=f'+1σ  ${mean_val+std_val:,.0f}')
ax1.axvline(mean_val-std_val, color=ORANGE, lw=1.5, linestyle=':')
ax1.set_title('Income Distribution with Standard Deviation Zones', color=TEXT, fontsize=12, fontweight='bold')
ax1.set_xlabel('Monthly Income (USD)')
ax1.set_ylabel('Frequency')
ax1.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax1.legend(fontsize=8.5, facecolor='#21262d', labelcolor=TEXT, ncol=4)

# Plot 2: Boxplot
ax2 = fig.add_subplot(gs[1, 0])
ax2.boxplot(inc, vert=True, patch_artist=True, widths=0.5,
    boxprops=dict(facecolor=BLUE, alpha=0.6, linewidth=1.5, edgecolor=BLUE),
    medianprops=dict(color=GREEN, linewidth=2.5),
    whiskerprops=dict(color=MUTED, linewidth=1.5, linestyle='--'),
    capprops=dict(color=MUTED, linewidth=2),
    flierprops=dict(marker='o', color=RED, alpha=0.4, markersize=3))
ax2.set_title('Boxplot (IQR)', color=TEXT, fontsize=11, fontweight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax2.set_xticks([])

# Plot 3: Empirical rule
ax3 = fig.add_subplot(gs[1, 1])
x = np.arange(3)
actual  = [within_1sigma, within_2sigma, within_3sigma]
normal_ = [68.27, 95.45, 99.73]
b1 = ax3.bar(x-0.175, actual,  0.35, label='This Dataset', color=BLUE,   alpha=0.85)
b2 = ax3.bar(x+0.175, normal_, 0.35, label='Normal Dist.', color=PURPLE, alpha=0.6)
for bar, val in zip(b1, actual):  ax3.text(bar.get_x()+0.175, bar.get_height()+0.5, f'{val:.1f}%',  ha='center', fontsize=8)
for bar, val in zip(b2, normal_): ax3.text(bar.get_x()+0.175, bar.get_height()+0.5, f'{val:.2f}%', ha='center', fontsize=8)
ax3.set_title('Empirical Rule Check', color=TEXT, fontsize=11, fontweight='bold')
ax3.set_xticks(x); ax3.set_xticklabels(['±1σ', '±2σ', '±3σ'])
ax3.set_ylim(0, 115); ax3.legend(fontsize=8, facecolor='#21262d', labelcolor=TEXT)

# Plot 4: Stats card
ax4 = fig.add_subplot(gs[1, 2]); ax4.axis('off')
ax4.set_title('Key Statistics', color=TEXT, fontsize=11, fontweight='bold')
stats = [
    ('Mean',     f'${mean_val:,.0f}',  BLUE),
    ('Median',   f'${inc.median():,.0f}', GREEN),
    ('Std Dev',  f'${std_val:,.0f}',   ORANGE),
    ('Variance', f'{var_val/1e9:.2f}B', PURPLE),
    ('IQR',      f'${iqr:,.0f}',       YELLOW),
    ('CV',       f'{cv:.1f}%',         RED),
    ('Skewness', f'{skewness:.4f}',    MUTED),
    ('Range',    '$15K – $250K',       MUTED),
]
ax4.set_xlim(0,1); ax4.set_ylim(0,1)
for i,(lbl,val,col) in enumerate(stats):
    y = 0.92 - i*0.115
    ax4.text(0.05, y, lbl, color=MUTED, fontsize=9, va='center')
    ax4.text(0.95, y, val, color=col,   fontsize=9.5, va='center', ha='right', fontweight='bold')
    ax4.axhline(y-0.048, color=BORDER, lw=0.5, alpha=0.7)

# Plot 5: CV gauge
ax5 = fig.add_subplot(gs[2, 0])
cats = ['Low (<15%)', 'Moderate (15–35%)', 'High (35–65%)', 'Very High (>65%)']
colors5 = [GREEN, YELLOW, ORANGE, RED]
ax5.barh(cats, [15, 20, 30, 20], color=colors5, alpha=0.5, height=0.5)
ax5.barh(['High (35–65%)'], [30], color=ORANGE, alpha=1.0, height=0.5, edgecolor=TEXT, lw=2)
ax5.text(31, 2, f'CV = {cv:.1f}%', color=ORANGE, fontsize=9, fontweight='bold', va='center')
ax5.set_title('Coefficient of Variation', color=TEXT, fontsize=11, fontweight='bold')
ax5.set_xlabel('Range (%)')

# Plot 6: Income segment donut
ax6 = fig.add_subplot(gs[2, 1])
bins_s = [0,50000,100000,150000,200000,300000]
lbls_s = ['<$50K','$50–100K','$100–150K','$150–200K','>$200K']
counts = pd.cut(inc, bins=bins_s, labels=lbls_s).value_counts().reindex(lbls_s)
wedges,texts,autos = ax6.pie(counts, labels=lbls_s, autopct='%1.1f%%',
    colors=[RED,ORANGE,BLUE,GREEN,PURPLE], startangle=90,
    textprops={'color':TEXT,'fontsize':8},
    wedgeprops={'linewidth':1.5,'edgecolor':BG}, pctdistance=0.75)
for at in autos: at.set_color(BG); at.set_fontsize(7.5); at.set_fontweight('bold')
ax6.add_patch(plt.Circle((0,0),0.45,color=PANEL))
ax6.text(0,0,'Income\nSegments',ha='center',va='center',color=TEXT,fontsize=8,fontweight='bold')
ax6.set_title('Income Segment Breakdown', color=TEXT, fontsize=11, fontweight='bold')

# Plot 7: CDF
ax7 = fig.add_subplot(gs[2, 2])
sorted_inc = np.sort(inc)
cdf = np.arange(1, len(sorted_inc)+1) / len(sorted_inc) * 100
ax7.plot(sorted_inc, cdf, color=BLUE, lw=2)
ax7.fill_between(sorted_inc, cdf, alpha=0.1, color=BLUE)
for xval, col in [(mean_val, RED), (mean_val-std_val, ORANGE), (mean_val+std_val, ORANGE)]:
    yval = (sorted_inc <= xval).mean()*100
    ax7.axvline(xval, color=col, lw=1.2, linestyle='--', alpha=0.8)
    ax7.scatter([xval], [yval], color=col, s=30, zorder=5)
ax7.set_title('Cumulative Distribution (CDF)', color=TEXT, fontsize=11, fontweight='bold')
ax7.set_xlabel('Monthly Income (USD)')
ax7.set_ylabel('Cumulative %')
ax7.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax7.set_ylim(0,105)

plt.savefig('visuals/income_dispersion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to visuals/income_dispersion.png')

## 6. Interpretation & Conclusions

| Metric | Value | Interpretation |
|:---|:---:|:---|
| **Mean** | $131,704 | Average monthly income |
| **Std Deviation** | $68,121 | Typical deviation from the mean |
| **Variance** | 4.64B | High absolute spread (in USD²) |
| **IQR** | $118,055 | The middle 50% spans a very wide range |
| **CV** | 51.7% | High relative dispersion — heterogeneous group |
| **Skewness** | 0.0108 | Nearly perfectly symmetric distribution |
| **Kurtosis** | −1.196 | Platykurtic — flatter than normal, no heavy tails |

### Key Insights

**1. Near-perfect symmetry:** A skewness of 0.0108 is remarkably close to zero, meaning income is distributed almost perfectly evenly around the mean — unusual for real income data, which typically skews right.

**2. High dispersion, no outliers:** The CV of 51.7% signals a highly heterogeneous group, yet there are zero statistical outliers beyond ±3σ. This indicates *organic* wide spread, not extreme individual cases.

**3. Platykurtic distribution:** Kurtosis of −1.196 means the distribution is flatter and has thinner tails than a normal curve. Income is spread broadly and evenly across all ranges, rather than clustering at the center.

**4. Empirical Rule deviation:** Only 57.5% of incomes fall within ±1σ (vs 68.27% expected for a normal distribution). This confirms the platykurtic, flat shape — data is more uniformly spread than a bell curve.

**5. Economic implication:** With incomes ranging from $15K to $250K and an IQR of $118K, this dataset represents an economically diverse population. A single marketing strategy or policy targeting 'the average consumer' would miss the majority of this audience.